# N017 · 数组扫描与原地覆盖

**目标：** 用读写指针维护已处理区间。

**先修：** N005, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 单遍扫描；原地契约；有效前缀；覆盖而非删除。

**配套讲解来源：** [同名逐章意见](../../comment/017_array_scans_inplace.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

数组题里有两大类高频操作："把数组里等于某个值的元素删掉"和"把数组里的零挪到末尾"。直观做法是每删一个元素就把后面整段往前搬一格，但那样删 k 个就要搬 k 次、每次最多搬 n 格，非常浪费。这章教你一个更聪明的通用套路：**双指针原地压缩**——用一个读指针扫一遍、一个写指针负责安置要保留的元素，一遍扫完就搞定，不多花一格额外空间。

两个具体到数字的例子：

- 输入 `a=[3,2,2,3]`、`val=3` → 输出 `k=2`，且数组前 2 格变成 `[2,2]` → 为什么？删掉两个 3 之后剩下的就是两个 2，题目契约只要求前 k 格正确，后面的内容无所谓。
- 输入 `a=[0,3,0,2,5]` → 输出数组变成 `[3,2,5,0,0]` → 为什么？把三个非零数 3、2、5 按原来的先后顺序压到前面，剩下的两格填零。

注意"移动零"额外要求保持非零元素的相对顺序（3 还得在 2 前面、2 还得在 5 前面），我们的写法天然满足这一点。

先把朴素做法和本章做法的差距摆出来（设数组长 n、要删的元素有 k 个）：

| 做法 | 过程 | 时间 |
|---|---|---|
| 朴素删除 | 每删一个，把后面整段往前搬一格 | 最坏 O(n×k)，接近 O(n²) |
| 读写指针压缩 | 一遍扫描，边读边往前写 | O(n) 一趟 |

n=10⁵、差不多一半要删时，朴素做法要搬约 25 亿次格，压缩做法只要约十万步——差了四五个数量级。

## 2. 基础知识：先读懂这些词

- **单遍扫描（one-pass scan）**：从左到右把数组只走一遍，走到哪处理到哪，绝不回头再走第二遍（移动零里补零的尾巴不算"重扫"，只是收尾填空）。
- **读指针 / 写指针（read/write pointers）**：两个下标变量。读指针负责"看下一个元素是什么"，写指针负责"下一个保留下来的元素该放哪"。两者从同一起点出发，读的一般跑在写的前面。
- **原地契约（in-place contract）**：题目要求不许开新数组，只能改动输入数组本身（通常只允许 O(1) 额外变量）。我们的做法完全符合：只在原数组上就地覆盖。
- **有效前缀（valid prefix）**：任何时刻，数组开头到写指针之间的那一段，是"已经处理完、保证正确"的保留元素区。删除类题目的答案就是"有效前缀的长度 k + 前 k 格内容"，k 之外的部分不属于答案。
- **覆盖而非删除（overwrite instead of delete）**：数组删元素并不是真的"抽走一格"，而是把后面的值往前覆盖。被覆盖的旧值不需要保存，因为它已经被判定为"不要的"或"已经复制到前面去了"。
- **稳定压缩（stable compaction）**：压缩之后保留下来的元素之间相对顺序不变。读写指针按从左到右的顺序搬运，谁先被读到谁先被安置，所以顺序天然保持稳定。

## 3. 思路推导：从小例子开始

- **Step 1：立好两个指针。** 读指针 `read` 和写指针 `write` 都从 0 出发。约定：`nums[0..write-1]` 是已完工的有效前缀。
- **Step 2：读指针逐格前进。** 每一步看 `nums[read]`：如果它是要保留的（不等于 val / 不等于 0），就把它复制到 `nums[write]`，然后写指针前进一格；如果它是要丢弃的，就什么都不做，只让读指针继续走。
- **Step 3：丢弃的元素自动被覆盖。** 被跳过的元素不需要显式删除，因为后续保留元素总会被往前写、把它盖掉；就算没人盖掉（它在有效长度之外），删除题也只看前 k 格，无所谓。
- **Step 4：移动零收尾填零。** 压缩完之后，有效前缀是全部非零数，数组尾部从 `write` 到末尾还留着旧值，我们要把它们统一改成 0，完成"零都挪到末尾"的契约。

**手算演示**（对应 “运行示例”部分 的表格，`a=[0,3,0,2,5]`，逐格走一遍压缩阶段）：

| 轮次 | read | 读取值 | 动作 | write | 有效前缀 |
|---|---|---|---|---|---|
| 1 | 0 | 0 | 是零，跳过 | 0 | `[]` |
| 2 | 1 | 3 | 保留：a[0]=3 | 1 | `[3]` |
| 3 | 2 | 0 | 是零，跳过 | 1 | `[3]` |
| 4 | 3 | 2 | 保留：a[1]=2 | 2 | `[3,2]` |
| 5 | 4 | 5 | 保留：a[2]=5 | 3 | `[3,2,5]` |

压缩完 write=3，数组此刻是 `[3,2,5,2,5]`（后两格是旧值残留）。收尾循环把下标 3、4 填成 0，最终得到 `[3,2,5,0,0]`。整个过程数组就完整地"当成了工作台"，一个新数组都没开。

**再演示 `remove_element`**（`a=[3,2,2,3]`、`val=3`）：

| 轮次 | read | 读取值 | 动作 | write | 数组状态（加粗为有效前缀） |
|---|---|---|---|---|---|
| 1 | 0 | 3 | 等于 val，跳过 | 0 | [] + [3,2,2,3] |
| 2 | 1 | 2 | 保留：a[0]=2 | 1 | [2] + [2,2,3] |
| 3 | 2 | 2 | 保留：a[1]=2 | 2 | [2,2] + [2,3] |
| 4 | 3 | 3 | 等于 val，跳过 | 2 | [2,2] + [2,3] |

四轮走完 write=2，前两格恰为 `[2,2]`——与第一节例子的预期完全一致。注意第 2 轮"a[0]=2"这一覆盖：它盖掉的是已经读过的 3，安全。

## 4. 核心代码：remove_element

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def remove_element(nums, val):
    write = 0
    for read in range(len(nums)):
        if nums[read] != val:
            nums[write] = nums[read]
            write += 1
    return write
```

### 逐段读懂代码

### 函数 1：`remove_element`（原地删除等于 val 的元素，返回有效长度）

先看前半段（初始化与筛选）：

```python
def remove_element(nums,val):
    write=0
    for read in range(len(nums)):
        if nums[read]!=val:
```

- `write=0`：写指针从 0 出发，也兼任"有效前缀长度"的计数器——现在有效前缀是空的。
- `for read in range(len(nums))`：读指针从左到右把整个数组扫一遍。
- `if nums[read]!=val:`：紧凑的筛选条件——只有"不等于目标值"的元素才配被保留；等于 val 的直接被这个 if 挡掉，什么也不发生。

再看后半段（复制与返回）：

```python
nums[write]=nums[read]
            write+=1
    return write
```

- `nums[write]=nums[read]`：把保留元素往前复制。注意当 `read==write` 时这一步是自己赋值给自己，看着多余但无害，省掉一个判断更简洁。
- `write+=1`：有效前缀向右扩一格。
- `return write`：循环结束时 write 恰好等于保留元素的个数，正是题目要的有效长度 k。

### 函数 2：`move_zeroes`（把所有零挪到末尾，无返回值）

压缩阶段与 `remove_element` 同构：

```python
def move_zeroes(nums):
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write] = nums[read]
            write += 1
```

- 前 5 行和 `remove_element` 几乎一模一样，只是筛选条件换成"非零才保留"——这就是"移动零 = 稳定压缩"的体现，压缩逻辑完全复用。

收尾阶段补零：

```python
for i in range(write,len(nums)):
        nums[i]=0
```

- `for i in range(write,len(nums)): nums[i]=0`：收尾的第二遍不是重新扫描，而是从写指针停下的位置开始，把剩下的尾巴统统清零。执行完后前 write 格是非零数（原顺序），后面全是 0。
- 注意这个函数没有 return——题目契约是"原地修改，无需返回"，调用方直接看数组本身。

## 5. 分段实现：按顺序运行

**本章接口：** `remove_element(nums, val)`、`move_zeroes(nums)`

### remove_element

In [1]:
def remove_element(nums, val):
    write = 0
    for read in range(len(nums)):
        if nums[read] != val:
            nums[write] = nums[read]
            write += 1
    return write

### move_zeroes

In [2]:
def move_zeroes(nums):
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write] = nums[read]
            write += 1
    for i in range(write, len(nums)):
        nums[i] = 0

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[0,3,0,2,5]; rows=[]; w=0
for r in range(len(a)):
    value=a[r]
    if value!=0: a[w]=value; w+=1
    rows.append((r,value,w,a[:w]))
show_table(['read','读取值','write','有效前缀'],rows)

read,读取值,write,有效前缀
0,0,0,[]
1,3,1,[3]
2,0,1,[3]
3,2,2,"[3, 2]"
4,5,3,"[3, 2, 5]"


## 7. 正确性、适用条件与复杂度

始终有 write≤read，写入不会覆盖未读元素；有效前缀按读取顺序增加，因此稳定且无遗漏。

**代价：** 两个算法 O(n) 时间、O(1) 辅助空间。

### 展开理解

**为什么是对的：** 关键的安全保障是"写指针永远不超过读指针"（write ≤ read 始终成立）。想想为什么：write 只有在读指针读过一格并确认保留之后才前进一格，所以 write 最多追平 read、永远不会跑到 read 前面去。既然写的位置永远不越过读的位置，被覆盖的就只能是"已经读过的旧值"，我们绝不可能覆盖掉还没读到的数据，这就是"不会丢元素"的保证。再一个保证是"不漏"：读指针把每一格都看了一遍，每个保留元素都会在它被读到的那一刻复制进有效前缀，而且复制顺序与读取顺序一致，所以相对顺序也被保住了。最后，move_zeroes 的收尾填零只动 `write` 之后的位置，那里全是被淘汰的旧值，动它不影响已完工的非零前缀。

**复杂度是多少：** 两个函数都是读指针走一遍、（最多）再补一遍尾巴，每个元素只被读写常数次，所以时间都是 O(n)；额外变量只有 read、write 两个下标（外加循环里的 i），空间是 O(1)。举个具体的数：n=10⁵ 时 O(n) 也就是十万次左右的操作，一瞬间就算完；对比朴素做法每删一个元素搬一次整个后缀、最坏 O(n²) 约一百亿次操作，差距是肉眼可见的。

**两个常见疑问，顺便说清：**

- 疑问一："后缀里的旧值（比如 `[3,2,5,2,5]` 尾巴上的 2,5）不用清理吗？"——不用。删除题的契约就是"前 k 格正确 + 返回 k"，k 之后是垃圾也无所谓；move_zeroes 才需要清尾巴，因为它的契约要求整个数组都是答案。
- 疑问二："能不能不覆盖、直接开个新列表装结果？"——功能上可以，但那就违反了"原地"契约、空间也升到 O(n)。面试或刷题里"原地"两个字就是硬性要求，读写指针正是为它准备的。

## 8. 单元测试：每个用例在检查什么

```python
a = [3, 2, 2, 3]

k = remove_element(a, 3)

assert k == 2 and a[:k] == [2, 2]

a = [3, 3]

assert remove_element(a, 3) == 0

from itertools import product

for n in range(6):
    for values in product([0, 1, 2], repeat=n):
        a = list(values)
        ref = [x for x in a if x] + [0] * a.count(0)
        move_zeroes(a)
        assert a == ref

assert remove_element([], 1) == 0

print('N017: 所有本章断言通过')
```

- 第 1 条是**正常值测试**（也是 LeetCode 27 的样例）：`[3,2,2,3]` 删 3 应得 k=2，且有效前缀 `a[:k]` 恰为 `[2,2]`。注意它只检查前 k 格——k 之后的内容按契约不算答案，这正是"有效长度之外的内容不属于删除题答案"的体现。
- 第 2 条是**全删空测试**（边界值）：`[3,3]` 全是待删值，返回 0，即有效前缀为空。这条检验 write 从 0 出发、一次都没前进时返回值是否正确。
- 第三段 for 是**穷举差分测试**：枚举元素取 0/1/2、长度 0~5 的全部数组（共 3⁰+3¹+...+3⁵=364 个），对每个数组用另一种写法构造参照答案 `ref=[x for x in a if x]+[0]*a.count(0)`（先挑出非零、再补上等量的零），然后断言 `move_zeroes` 的结果与参照逐格相等。参照是"开新列表重排"的思路，与被测的"原地覆盖"思路完全不同，对得上才有说服力。
- `assert remove_element([],1)==0`：**空输入测试**。空数组没有元素可删，长度自然为 0，这条保证循环一次不转、直接返回 write=0 的路径也是对的。

In [4]:
a = [3, 2, 2, 3]

k = remove_element(a, 3)

assert k == 2 and a[:k] == [2, 2]

a = [3, 3]

assert remove_element(a, 3) == 0

from itertools import product

for n in range(6):
    for values in product([0, 1, 2], repeat=n):
        a = list(values)
        ref = [x for x in a if x] + [0] * a.count(0)
        move_zeroes(a)
        assert a == ref

assert remove_element([], 1) == 0

print('N017: 所有本章断言通过')

N017: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 要求保持非零元素相对顺序。

**练习 2：** 把删除特定值推广到谓词过滤。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（保持非零元素相对顺序的移动零）**：思路方向——其实本章 `move_zeroes` 已经做到了稳定，这题让你解释"为什么"。提示：回顾读写指针"谁先被读到谁先被安置"的性质，再用 `[0,3,0,2,5]` 手算一遍（就用第三节的表格），观察 3、2、5 的先后关系从未改变。也可以想想：如果改成"交换 nums[read] 和 nums[write]"的写法，稳定性还保得住吗？（提示：交换版在特定输入下会把后面的零换到前面非零区，想想哪一步会出问题。）
- **练习 2（把删除特定值推广到谓词过滤）**：思路方向——把 `nums[read]!=val` 这个具体条件抽象成一个函数参数 `pred(x)`，只有 `pred(nums[read])` 为真才保留。提示：函数签名可以设计成 `filter_inplace(nums, pred)`，返回有效长度；手算例子可以用"保留偶数"这种谓词，如 `[1,2,3,4]` 过滤偶数后有效前缀是 `[2,4]`、返回 2。写完想想它与 Python 内置 `filter` 的区别——那个会开新序列，你这个是原地的。

两道练习验收时都要求：输入、输出、边界三件事写清楚，并且能口头回答"write 什么时候会等于 read、什么时候会落后"——答得上来，说明你真的在脑子里跑过这两个指针。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def remove_element(nums, val):
    write = 0
    for read in range(len(nums)):
        if nums[read] != val:
            nums[write] = nums[read]
            write += 1
    return write

def move_zeroes(nums):
    write = 0
    for read in range(len(nums)):
        if nums[read] != 0:
            nums[write] = nums[read]
            write += 1
    for i in range(write, len(nums)):
        nums[i] = 0

# 示例与回归测试
a = [3, 2, 2, 3]

k = remove_element(a, 3)

assert k == 2 and a[:k] == [2, 2]

a = [3, 3]

assert remove_element(a, 3) == 0

from itertools import product

for n in range(6):
    for values in product([0, 1, 2], repeat=n):
        a = list(values)
        ref = [x for x in a if x] + [0] * a.count(0)
        move_zeroes(a)
        assert a == ref

assert remove_element([], 1) == 0

print('N017: 所有本章断言通过')

N017: 所有本章断言通过


## 11. 代表题与迁移

- **27. Remove Element（移除元素）**：这题就是本章 `remove_element` 的原型——原地删除等于 val 的元素并返回新长度，读写指针压缩是标准解法。
- **283. Move Zeroes（移动零）**：这题就是本章 `move_zeroes` 的原型——把零挪到末尾且保持非零元素相对顺序，练的是"稳定压缩 + 尾部填零"这个组合。
- 迁移建议：这个"读写指针维护有效前缀"的骨架以后还会反复出现（去重、快排的 partition、字符串压缩都是它的变体），值得吃透到能默写的程度。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。